# Archive — superseded design

**These cells do not reproduce anything in the letter.** They are kept for
transparency about how the study developed.

They belong to an earlier design that targeted ERA5-Land `temperature_2m`
rather than MODIS night LST. That target has almost no intra-urban
structure — roughly seven distinct values across thousands of nodes, with a
spatial standard deviation of 0.09–0.23 °C — so the apparent spatial skill
those experiments measured was largely the model recovering which coarse
reanalysis cell each node occupied. The correlation between building cover
and target anomaly even changed sign between two Indian cities in the same
season.

Conclusions drawn here were not carried into the letter.


In [ ]:
# ============================================================
# EVALUATE SAVED CHECKPOINTS -- no retraining
# ------------------------------------------------------------
# Cell 9 saves each arm as it finishes:
#     {OUT_ROOT}/{city}/{month}_Models/{month}_{arm}.pt
# containing state_dict, feature_mask, alpha, gamma, cover_t, ndvi_t,
# temp_mean and temp_std. That is everything needed to rebuild the
# model and its prep() wrapper.
#
# prepare() still runs, to rebuild the loaders and the graph -- but
# that is seconds, not the 4.4 min of training per arm.
#
# Requires prediction_quality() to be defined (paste that cell first).
# ============================================================

# ============================================================
# IS THE MODEL ACTUALLY PREDICTING ANYTHING?
# ------------------------------------------------------------
# RMSE alone cannot answer this. Two failure modes hide inside it:
#
# A. NO SKILL OVER PERSISTENCE. At horizon=1, "next hour = this hour"
#    is a strong baseline. The persistence figure printed by prepare()
#    (1.241 C on Blr April) covers ALL 336 timesteps, while the model
#    RMSE (1.379 C) is the test window only -- so they are not
#    comparable as printed. This computes persistence ON THE TEST SET,
#    which is the fair comparison.
#
# B. NO SPATIAL SKILL. Temperature varies ~1-2 C across the city at a
#    given hour but 10+ C over the diurnal cycle, so RMSE is dominated
#    by the time series. A model that predicts the city-wide mean each
#    hour and nothing spatial scores well on RMSE while having none of
#    the structure a GNN is supposed to provide. The predicted-vs-
#    actual scatter hides this too, because it is dominated by the
#    same diurnal spread.
#
#    The decomposition below splits each timestep into
#        field mean  +  spatial anomaly
#    and scores them separately. The spatial anomaly correlation is
#    the number that says whether the graph is earning its place.
#
# Run on a trained model. Cheap -- one pass over the test loader.
# ============================================================

def prediction_quality(model, loader, temp_mean, temp_std, tag=""):
    model.eval()
    P, Y, PERS = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            fp, tp, tr = model(batch.x, batch.edge_index)
            P.append(fp.cpu().numpy())
            Y.append(batch.y.cpu().numpy())
            # persistence: predict t+h as the temperature at the last
            # hour of the input window. Feature 0 is temperature_2m,
            # same normalisation as y, so directly comparable.
            PERS.append(batch.x[:, -1, 0].cpu().numpy())

    # [T_windows, N_nodes], still in normalised units
    P = np.stack(P); Y = np.stack(Y); PERS = np.stack(PERS)
    to_c = temp_std                      # normalised -> degrees

    def rmse(a, b):
        return float(np.sqrt(np.mean((a - b) ** 2)) * to_c)

    r_model = rmse(P, Y)
    r_pers = rmse(PERS, Y)
    # climatology: each node's own test-period mean, broadcast over time
    clim = np.repeat(Y.mean(axis=0, keepdims=True), Y.shape[0], axis=0)
    r_clim = rmse(clim, Y)

    print(f"\n{'=' * 66}")
    print(f"PREDICTION QUALITY {tag}")
    print(f"{'=' * 66}")
    print(f"  test windows {P.shape[0]}  nodes {P.shape[1]}")
    print(f"\n  RMSE, all on the SAME test set:")
    print(f"    model                {r_model:>7.3f} C")
    print(f"    persistence (t+1=t)  {r_pers:>7.3f} C")
    print(f"    climatology (node mean){r_clim:>5.3f} C")

    skill_p = 1 - (r_model / r_pers) ** 2
    skill_c = 1 - (r_model / r_clim) ** 2
    print(f"\n  skill score (1 - MSE_model/MSE_baseline; >0 means better):")
    print(f"    vs persistence  {skill_p:+.3f}"
          f"   {'OK' if skill_p > 0 else '<-- NO SKILL over persistence'}")
    print(f"    vs climatology  {skill_c:+.3f}"
          f"   {'OK' if skill_c > 0 else '<-- NO SKILL over node means'}")

    # ---- spatial vs temporal decomposition ----------------------
    # Each timestep: field mean + spatial anomaly.
    Ym, Pm = Y.mean(axis=1, keepdims=True), P.mean(axis=1, keepdims=True)
    Ya, Pa = Y - Ym, P - Pm

    r_mean = rmse(Pm, Ym)                # error in the city-wide level
    r_anom = rmse(Pa, Ya)                # error in the spatial pattern
    print(f"\n  error decomposition:")
    print(f"    field-mean component   {r_mean:>7.3f} C")
    print(f"    spatial-anomaly comp.  {r_anom:>7.3f} C")
    print(f"    actual spatial spread  {float(Ya.std() * to_c):>7.3f} C"
          f"   (std of true anomalies)")
    print(f"    predicted sp. spread   {float(Pa.std() * to_c):>7.3f} C")

    # Per-timestep spatial correlation: does the predicted pattern
    # match the actual pattern at each hour?
    rs = []
    for t in range(Ya.shape[0]):
        if Ya[t].std() > 1e-9 and Pa[t].std() > 1e-9:
            rs.append(np.corrcoef(Ya[t], Pa[t])[0, 1])
    rs = np.array(rs)
    print(f"\n  SPATIAL SKILL -- corr(predicted anomaly, actual anomaly)")
    print(f"    per-timestep mean {rs.mean():+.3f}   median {np.median(rs):+.3f}"
          f"   min {rs.min():+.3f}   max {rs.max():+.3f}")
    if rs.mean() < 0.2:
        print("    !! The model has little spatial skill. It is tracking the")
        print("       diurnal cycle, not the spatial pattern -- which is what")
        print("       the graph is supposed to provide.")
    elif rs.mean() < 0.5:
        print("    Moderate spatial skill.")
    else:
        print("    Strong spatial skill.")

    # Variance ratio: how much of the predicted spatial spread is real?
    ratio = float(Pa.std() / Ya.std()) if Ya.std() > 0 else float('nan')
    print(f"\n    predicted/actual spatial spread ratio {ratio:.3f}")
    if ratio < 0.5:
        print("    !! Predictions are spatially FLATTER than reality --")
        print("       the model is hedging toward the field mean.")

    # ---- head contributions -------------------------------------
    PH, TR = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, tp, tr = model(batch.x, batch.edge_index)
            PH.append(tp.cpu().numpy()); TR.append(tr.cpu().numpy())
    PH, TR = np.stack(PH), np.stack(TR)
    print(f"\n  head contributions (degrees):")
    print(f"    t_phys  std {float(PH.std() * to_c):>6.3f}   "
          f"range [{float(PH.min()*to_c):+.2f}, {float(PH.max()*to_c):+.2f}]")
    print(f"    t_res   std {float(TR.std() * to_c):>6.3f}   "
          f"range [{float(TR.min()*to_c):+.2f}, {float(TR.max()*to_c):+.2f}]")
    # How much of the SPATIAL pattern comes from the residual head?
    TRa = TR - TR.mean(axis=1, keepdims=True)
    frac = float(TRa.std() / Pa.std()) if Pa.std() > 0 else float('nan')
    print(f"    residual head supplies {frac:.1%} of the predicted "
          f"spatial spread")

    return dict(rmse=r_model, rmse_persistence=r_pers, rmse_climatology=r_clim,
                skill_vs_persistence=skill_p, skill_vs_climatology=skill_c,
                rmse_field_mean=r_mean, rmse_spatial_anom=r_anom,
                spatial_corr_mean=float(rs.mean()),
                spatial_corr_median=float(np.median(rs)),
                spatial_spread_ratio=ratio,
                res_spatial_share=frac)


# ------------------------------------------------------------
# USAGE, after training a model in cell 9:
#
#     q = prediction_quality(m, L['test'], D['temp_mean'], D['temp_std'],
#                            tag=f"{city} {month} {tag}")
#
# and add q to the row you append to val_rows.
#
# READ IN THIS ORDER:
#   1. skill vs persistence. Negative at horizon=1 means the model is
#      not forecasting -- everything downstream is moot.
#   2. spatial correlation. Near zero means the graph is not
#      contributing, whatever the RMSE says.
#   3. spatial spread ratio. Well below 1 means predictions are
#      flattened toward the field mean, which is what a model does
#      when it cannot resolve the spatial pattern and minimises
#      squared error by hedging.
# ------------------------------------------------------------

import glob

rows = []

for cfg in CONFIGS:
    city, month = cfg['city'], cfg['month']
    ckpts = sorted(glob.glob(f"{OUT_ROOT}/{city}/{month}_Models/{month}_*.pt"))
    if not ckpts:
        print(f"\n### {city} {month}: no checkpoints yet, skipping")
        continue

    print(f"\n### {city} — {month}   ({len(ckpts)} checkpoint(s))")
    D = prepare(cfg, HP)
    node_ids = D['gdf'][ID_COL].values
    try:
        lst = load_lst(cfg, node_ids)
        refs = lst_reference_correlations(cfg, lst, node_ids)
        r_cover = refs['bld_cover']['pearson']
        r_ndbi = refs['NDBI']['pearson']
    except KeyError as e:
        print(f"  LST unavailable: {e}")
        lst, r_cover, r_ndbi = None, float('nan'), float('nan')

    for path in ckpts:
        ck = torch.load(path, map_location=device)
        arm = ck.get('arm', os.path.basename(path))

        # Guard: a checkpoint trained on different data would give
        # meaningless numbers against the current loaders.
        if abs(ck['temp_std'] - D['temp_std']) > 1e-3:
            print(f"  !! {arm}: checkpoint temp_std {ck['temp_std']:.4f} != "
                  f"current {D['temp_std']:.4f} -- trained on DIFFERENT data. "
                  f"Skipping.")
            continue

        model = PhyRes(len(FEATURE_COLS), HP['spatial_dim'],
                       HP['lstm_hidden'], HP['dropout']).to(device)
        model.load_state_dict(ck['state_dict'])
        fmask = ck['feature_mask'].to(device)

        def prep(x, _m=fmask):        # bind by value, not by closure
            return x * _m

        m = MaskedModel(model, prep).to(device)
        L = D['pruned']

        q = prediction_quality(m, L['test'], ck['temp_mean'], ck['temp_std'],
                               tag=f"{city} {month} — {arm}"
                                   f"  (alpha={ck.get('alpha')}, gamma={ck.get('gamma')})")

        # LST agreement, same as cell 9
        r_lst = rs_lst = float('nan')
        if lst is not None:
            res_map = mean_residual_map(m, L['test'])
            ok = np.isfinite(lst) & np.isfinite(res_map)
            if res_map.std() > 1e-12 and ok.sum() >= 50:
                r_lst = float(np.corrcoef(res_map[ok], lst[ok])[0, 1])
                rs_lst = float(stats.spearmanr(res_map[ok], lst[ok]).statistic)

        rows.append(dict(city=city, month=month, arm=arm,
                         alpha=ck.get('alpha'), gamma=ck.get('gamma'),
                         r_res_lst=r_lst, r_res_lst_spearman=rs_lst,
                         r_bldg_lst=r_cover, r_ndbi_lst=r_ndbi, **q))

        del model, m
        if device.type == 'cuda':
            torch.cuda.empty_cache()

if rows:
    q = pd.DataFrame(rows)
    q.to_csv(f"{OUT_ROOT}/prediction_quality.csv", index=False)

    print("\n" + "=" * 78)
    print("SUMMARY — the three numbers that decide whether the model works")
    print("=" * 78)
    cols = ['city', 'month', 'arm', 'rmse', 'rmse_persistence',
            'skill_vs_persistence', 'spatial_corr_mean',
            'spatial_spread_ratio', 'res_spatial_share', 'r_res_lst']
    print(q[cols].round(3).to_string(index=False))

    bad_skill = q[q.skill_vs_persistence <= 0]
    bad_spatial = q[q.spatial_corr_mean < 0.2]
    print()
    if len(bad_skill):
        print(f"!! {len(bad_skill)}/{len(q)} arms have NO SKILL over persistence.")
        print("   At horizon=1 that means the model is not forecasting; it is")
        print("   reproducing the last observed hour, or worse. Nothing")
        print("   downstream of that is meaningful.")
    if len(bad_spatial):
        print(f"!! {len(bad_spatial)}/{len(q)} arms have spatial correlation < 0.2.")
        print("   The graph is not contributing to the spatial pattern,")
        print("   whatever the RMSE says.")
    if not len(bad_skill) and not len(bad_spatial):
        print("Both checks pass on every arm.")
else:
    print("\nNo checkpoints evaluated.")

In [ ]:
# ============================================================
# GAMMA SWEEP -- does the push-pull prior help or hurt?
# ------------------------------------------------------------
# On Bengaluru April at gamma=0.03, push-pull is 28.15% of the
# objective (mask normalisation multiplied it by ~11x) and:
#     r(t_res, LST) = +0.451
#     r(NDBI,  LST) = +0.684     <- a raw model INPUT beats t_res
#     r(cover, LST) = +0.068     <- what push-pull is anchored on
#
# The prior forces t_res UP wherever bld_cover > 0.4. In this config
# bld_cover barely correlates with observed surface heat, so a
# heavily weighted prior should drag t_res toward a pattern that does
# not match LST.
#
# PREDICTION: if the prior is degrading the validation metric,
# r(t_res, LST) RISES as gamma falls, and is highest at gamma=0.
# If instead the prior encodes something real, r(t_res, LST) peaks at
# some gamma > 0.
#
# gamma=0 is the NoPushPull arm: it also re-tests the rBldg
# circularity claim on corrected data. rBldg should collapse toward 0.
#
# Runs ONE config by default -- the sweep is the point, not coverage.
# ~4.5 min per point.
# ============================================================

SWEEP_CFG = CONFIGS[0]                       # Bengaluru April
GAMMAS = [0.0, 0.0027, 0.01, 0.03, 0.1]
# 0.0027 ~= 0.03 / 11.2, i.e. the effective weight the OLD
# unnormalised gamma=0.03 applied. Included so the sweep contains a
# point comparable to the earlier runs.

print("=" * 78)
print(f"GAMMA SWEEP — {SWEEP_CFG['city']} {SWEEP_CFG['month']}  (alpha={ALPHA})")
print("=" * 78)

D = prepare(SWEEP_CFG, HP)
node_ids = D['gdf'][ID_COL].values
lst = load_lst(SWEEP_CFG, node_ids)
refs = lst_reference_correlations(SWEEP_CFG, lst, node_ids)
r_cover = refs['bld_cover']['pearson']
r_ndbi = refs['NDBI']['pearson']
print(f"\nbaselines:  r(cover,LST) {r_cover:+.3f}   r(NDBI,LST) {r_ndbi:+.3f}")
report_mask_coverage(L['train'], tag, D['temp_std'])

rows = []
for g in GAMMAS:
    set_seed(HP['seed'])
    fmask = make_feature_mask([])
    L = D['pruned']

    raw = PhyRes(len(FEATURE_COLS), HP['spatial_dim'],
                 HP['lstm_hidden'], HP['dropout']).to(device)
    raw, prep = train_masked(raw, L['train'], L['val'], HP['epochs'], lr=8e-4,
                             alpha=ALPHA, gamma=g, weight_decay=1e-4, clip=0.5,
                             tag=f"gamma={g}", temp_std=D['temp_std'], feat_mask=fmask)
    m = MaskedModel(raw, prep).to(device)

    share = loss_breakdown(m, L['val'], ALPHA, GAMMA, D['temp_std'], tag)
    pp_share = 100 * share['pushpull_w'] / share['total'] if share['total'] else 0.0

    st = {}
    rmse, rbldg = evaluate_masked(m, L['test'], prep,
                                  D['temp_mean'], D['temp_std'], stats_out=st)

    res_map = mean_residual_map(m, L['test'])
    ok = np.isfinite(lst) & np.isfinite(res_map)
    if res_map.std() < 1e-12 or ok.sum() < 50:
        r_lst = rs_lst = float('nan')
    else:
        r_lst = float(np.corrcoef(res_map[ok], lst[ok])[0, 1])
        rs_lst = float(stats.spearmanr(res_map[ok], lst[ok]).statistic)

    rows.append(dict(gamma=g, pushpull_pct=pp_share, rmse=rmse,
                     rBldg=rbldg, rBldg_spearman=st['rBldg_spearman'],
                     res_std=st['res_std'],
                     r_res_lst=r_lst, r_res_lst_spearman=rs_lst,
                     gain_vs_cover=abs(r_lst) - abs(r_cover),
                     gain_vs_ndbi=abs(r_lst) - abs(r_ndbi)))
    print(f"  -> gamma={g}: RMSE {rmse:.3f}  rBldg {rbldg:+.3f}  "
          f"r(t_res,LST) {r_lst:+.3f}\n")

    del raw, m
    if device.type == 'cuda':
        torch.cuda.empty_cache()

s = pd.DataFrame(rows)
s.to_csv(f"{OUT_ROOT}/gamma_sweep.csv", index=False)

print("=" * 78)
print(s.round(4).to_string(index=False))
print("=" * 78)

best = s.loc[s.r_res_lst.abs().idxmax()]
print(f"""
r(t_res,LST) is highest at gamma={best.gamma} ({best.r_res_lst:+.3f}).

  highest at gamma=0        -> the push-pull prior DEGRADES agreement
                               with observed surface heat. It is
                               anchored on bld_cover, which correlates
                               {r_cover:+.3f} with LST here.
  peaks at some gamma>0     -> the prior encodes something real; report
                               the peak and the sweep.
  flat across gamma         -> the prior is inert at these weights.

Also read rBldg down the gamma column. If it collapses toward 0 at
gamma=0, that re-confirms on corrected data what the earlier session
found on the broken column: rBldg measures optimiser compliance with
the prior, not discovered structure. It should not be presented as
evidence the model found anything.

Note the noise floor: differences below ~0.185 C in RMSE are not real
(3 seeds, measured earlier). No equivalent floor has been measured for
r(t_res,LST) -- worth doing with 3 seeds at the best and worst gamma
before drawing a conclusion from the shape of this curve.
""")

In [ ]:
# ============================================================
# DOES THE TARGET CONTAIN AN URBAN HEAT SIGNAL AT ALL?
# ------------------------------------------------------------
# The model predicts ERA5-Land temperature_2m, native resolution
# ~0.1 deg (~11 km). The Bengaluru rectangle is 0.30 x 0.25 deg, so
# roughly 8 native cells cover 3,679 nodes at 500 m spacing.
# reduceRegions at scale 30 gives nodes inside one cell an identical
# value and nodes on a boundary a blend -- which is why 413 distinct
# values appear rather than 8.
#
# The question that decides the project is NOT how many distinct
# values exist. It is whether the target's SPATIAL pattern relates to
# urban form. Three tests:
#
#   A. PLATEAU STRUCTURE. If a handful of values cover most nodes,
#      the field is a resampled coarse grid, not a city-scale field.
#
#   B. URBAN SIGNAL. Correlate the target's spatial anomaly against
#      bld_cover, NDBI and observed LST. If these are ~0, ERA5
#      temperature_2m carries no urban heat structure at this scale
#      and no model can learn UHI from it -- the spatial correlations
#      of 0.13-0.51 would then be the model recovering WHICH ERA5
#      CELL each node sits in, which is geometry, not physics.
#
#   C. SCALE COMPARISON. Spatial spread of the target (~0.22-0.29 C)
#      against the spread of LST over the same nodes. LST is a real
#      500 m surface observation; if it varies 10x more spatially,
#      that gap is the ceiling on what the target could ever show.
# ============================================================
import os
import numpy as np
import pandas as pd
from scipy import stats

LOCAL_DIR = '/content/data'
FILES = [
    ('Blr_April',    f'{LOCAL_DIR}/Bangalore_Summer_April_PhyRes_v33.csv'),
    ('Blr_December', f'{LOCAL_DIR}/Bangalore_Winter_Dec_PhyRes_v33.csv'),
    ('Hyd_April',    f'{LOCAL_DIR}/Hyderabad_Summer_April_PhyRes_v33.csv'),
    ('Hyd_December', f'{LOCAL_DIR}/Hyderabad_Winter_Dec_PhyRes_v33.csv'),
]
ID_COL, TIME_COL, T_COL = 'id', 'timestamp', 'temperature_2m'

for tag, path in FILES:
    if not os.path.exists(path):
        print(f"{tag}: MISSING\n"); continue
    print("=" * 72)
    print(tag)
    print("=" * 72)

    d = pd.read_csv(path)
    if np.issubdtype(d[TIME_COL].dtype, np.number):
        d[TIME_COL] = pd.to_datetime(d[TIME_COL], unit='ms')
    d = d.dropna(subset=[T_COL, 'bld_cover', 'NDBI', 'LST_landsat'])

    # ---- A. plateau structure at one timestep -------------------
    t0 = d[d[TIME_COL] == d[TIME_COL].iloc[0]]
    vc = t0[T_COL].value_counts()
    n = len(t0)
    print(f"\nA. one timestep: {n:,} nodes, {t0[T_COL].nunique()} distinct values")
    print(f"   top 8 values by node count:")
    for v, c in vc.head(8).items():
        print(f"     {v:.6f}  ->  {c:>5} nodes ({100*c/n:>5.1f}%)")
    print(f"   top 8 values cover {100*vc.head(8).sum()/n:.1f}% of nodes")
    print(f"   values held by only 1 node: {int((vc == 1).sum())}")
    print("   -> a few values covering most nodes = resampled coarse grid;")
    print("      the singletons are cells straddling ERA5 boundaries.")

    # ---- B. does the target's spatial pattern track urban form? --
    # Per-node time-mean, then remove the field mean: what is left is
    # purely the spatial pattern of the target.
    g = d.groupby(ID_COL)[[T_COL, 'bld_cover', 'NDBI', 'NDVI',
                           'LST_landsat']].mean()
    anom = g[T_COL] - g[T_COL].mean()

    print(f"\nB. target spatial anomaly vs urban form:")
    print(f"   {'against':<14}{'Pearson':>10}{'Spearman':>10}")
    for col in ('bld_cover', 'NDBI', 'NDVI', 'LST_landsat'):
        p = float(np.corrcoef(anom, g[col])[0, 1])
        s = float(stats.spearmanr(anom, g[col]).statistic)
        print(f"   {col:<14}{p:>+10.3f}{s:>+10.3f}")

    # The decisive one: does the target agree with the independent
    # surface observation over the same nodes?
    r_lst = float(np.corrcoef(anom, g['LST_landsat'])[0, 1])

    # ---- C. how much spatial variation is there to learn? -------
    t_sd = float(g[T_COL].std())
    l_sd = float(g['LST_landsat'].std())
    # temporal spread for scale
    temporal_sd = float(d.groupby(TIME_COL)[T_COL].mean().std())
    print(f"\nC. spatial vs temporal variation in the target:")
    print(f"   target spatial std (across nodes)   {t_sd:>6.3f} C")
    print(f"   target temporal std (field mean)    {temporal_sd:>6.3f} C")
    print(f"   ratio spatial/temporal              {t_sd/temporal_sd:>6.3f}")
    print(f"   LST spatial std (same nodes)        {l_sd:>6.3f} C"
          f"   [{l_sd/t_sd:.1f}x the target]")

    print(f"\n   VERDICT: ", end="")
    if abs(r_lst) < 0.15:
        print("the target's spatial pattern does NOT track observed")
        print("   surface temperature. ERA5-Land temperature_2m at ~11 km")
        print("   carries no urban heat structure at 500 m. A GNN cannot")
        print("   learn UHI from this target regardless of architecture,")
        print("   and reported spatial skill is the model recovering which")
        print("   ERA5 cell a node falls in.")
    elif abs(r_lst) < 0.4:
        print("weak agreement with observed surface temperature.")
        print("   Some urban signal may survive downscaling, but it is small")
        print("   relative to the coarse-grid structure.")
    else:
        print("the target does track observed surface temperature.")
        print("   Urban signal survives in ERA5 at this scale.")
    print()

In [ ]:
# ============================================================
# DOES t_res CARRY ANYTHING ITS INPUTS DON'T?
# ------------------------------------------------------------
# r(t_res, LST) is 0.42-0.49 in three configs. But NDBI -- a model
# INPUT -- reaches 0.68-0.81 against the same LST. So the raw
# correlation cannot tell us whether the residual head discovered
# anything or simply re-expressed a spectral index it was handed.
#
# Two tests, both on saved checkpoints (no retraining):
#
#   A. PARTIAL CORRELATION. r(t_res, LST | NDBI), then controlling
#      for every morphological input at once. If these collapse to
#      ~0, t_res is a repackaging of its inputs.
#
#   B. INCREMENTAL R². Fit LST ~ inputs, then LST ~ inputs + t_res.
#      The gain in R² is the share of surface-heat variance that
#      t_res explains and the inputs alone cannot. This is the
#      version to put in a paper: it answers "what does the model
#      add?" in one number a reviewer can check.
#
# Requires: prediction_quality / eval_checkpoints context (CONFIGS,
# prepare, load_lst, MaskedModel, mean_residual_map).
# ============================================================
import glob
import numpy as np
import pandas as pd
from scipy import stats
import torch.nn.functional as F

CTRL_SETS = {
    'NDBI only':        ['NDBI'],
    'NDBI+NDVI':        ['NDBI', 'NDVI'],
    'all morphology':   ['NDBI', 'NDVI', 'bld_cover', 'bld_height',
                         'traffic_no2_proxy'],
}


def partial_corr(x, y, Z):
    """corr(x, y) after regressing both on the columns of Z."""
    Z1 = np.column_stack([np.ones(len(Z)), Z])
    rx = x - Z1 @ np.linalg.lstsq(Z1, x, rcond=None)[0]
    ry = y - Z1 @ np.linalg.lstsq(Z1, y, rcond=None)[0]
    if rx.std() < 1e-12 or ry.std() < 1e-12:
        return float('nan')
    return float(np.corrcoef(rx, ry)[0, 1])


def r2(y, X):
    X1 = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(X1, y, rcond=None)[0]
    resid = y - X1 @ beta
    return 1 - float(resid.var() / y.var())


rows = []
for cfg in CONFIGS:
    city, month = cfg['city'], cfg['month']
    ckpts = sorted(glob.glob(f"{OUT_ROOT}/{city}/{month}_Models/{month}_*.pt"))
    if not ckpts:
        continue

    D = prepare(cfg, HP)
    node_ids = D['gdf'][ID_COL].values
    lst = load_lst(cfg, node_ids)

    # node-level feature table, aligned to the same node set
    raw = pd.read_csv(cfg['csv'])
    raw[ID_COL] = _align_ids(raw[ID_COL])
    feats = (raw.groupby(ID_COL)[['NDBI', 'NDVI', 'bld_cover', 'bld_height',
                                  'traffic_no2_proxy']]
             .mean().reindex(node_ids))

    for path in ckpts:
        ck = torch.load(path, map_location=device)
        arm = ck.get('arm', os.path.basename(path))
        if abs(ck['temp_std'] - D['temp_std']) > 1e-3:
            continue

        model = PhyRes(len(FEATURE_COLS), HP['spatial_dim'],
                       HP['lstm_hidden'], HP['dropout']).to(device)
        model.load_state_dict(ck['state_dict'])
        fmask = ck['feature_mask'].to(device)
        m = MaskedModel(model, lambda x, _f=fmask: x * _f).to(device)

        t_res = mean_residual_map(m, D['pruned']['test'])
        ok = (np.isfinite(lst) & np.isfinite(t_res)
              & feats.notna().all(axis=1).values)
        if ok.sum() < 50 or np.std(t_res[ok]) < 1e-12:
            print(f"  {city} {month} {arm}: degenerate, skipping")
            continue

        y, tr = lst[ok], t_res[ok]
        FT = feats[ok]                     # NOT F -- that is torch.nn.functional

        row = dict(city=city, month=month, arm=arm,
                   r_raw=float(np.corrcoef(tr, y)[0, 1]),
                   r_ndbi=float(np.corrcoef(FT["NDBI"], y)[0, 1]))
        for name, cols in CTRL_SETS.items():
            row[f"partial|{name}"] = partial_corr(tr, y, FT[cols].values)

        base = r2(y, FT[CTRL_SETS['all morphology']].values)
        full = r2(y, np.column_stack([FT[CTRL_SETS['all morphology']].values, tr]))
        row['R2_inputs'] = base
        row['R2_inputs+tres'] = full
        row['R2_gain'] = full - base
        rows.append(row)

        del model, m
        if device.type == 'cuda':
            torch.cuda.empty_cache()

q = pd.DataFrame(rows)
q.to_csv(f"{OUT_ROOT}/tres_contribution.csv", index=False)

print("\n" + "=" * 86)
print("A. PARTIAL CORRELATION of t_res with LST, controlling for model inputs")
print("=" * 86)
print(q[['city', 'month', 'arm', 'r_raw', 'r_ndbi',
         'partial|NDBI only', 'partial|NDBI+NDVI',
         'partial|all morphology']].round(3).to_string(index=False))

print("\n" + "=" * 86)
print("B. INCREMENTAL R2 -- variance in LST explained by t_res beyond the inputs")
print("=" * 86)
print(q[['city', 'month', 'arm', 'R2_inputs', 'R2_inputs+tres', 'R2_gain']]
      .round(4).to_string(index=False))

g = q['R2_gain'].abs()
p = q['partial|all morphology'].abs()
print(f"""
median R2 gain          {g.median():.4f}
median |partial| (all)  {p.median():.3f}

  gain > ~0.02 and partial > ~0.2
      -> t_res explains surface heat variance the raw inputs do not.
         That is a real contribution and the downscaling framing has
         a result at its centre.

  gain ~ 0 and partial ~ 0
      -> t_res is a linear re-expression of inputs the model was
         already given. The residual field would not be a discovery,
         and a reviewer running this exact check would find the same.
""")

In [ ]:
# ============================================================
# DOES SPATIAL DEPENDENCE HELP? — GNN vs regression on LST
# ------------------------------------------------------------
# New framing. The old target (ERA5 temperature_2m) had no
# intra-urban structure: ~8 distinct values over 3,679 nodes, spatial
# std 0.09-0.23 C, and its correlation with bld_cover flipped sign
# between cities. Landsat LST over the SAME nodes has spatial std
# 1.35-2.30 C and behaves consistently. So predict that instead.
#
# The question a graph can actually answer:
#   does modelling spatial dependence beat treating each node
#   independently?
#
# BASELINES, in increasing order of what they concede:
#   1. mean            — R2 = 0 by construction, sanity floor.
#   2. OLS             — own features only. Already known to reach
#                        R2 0.51-0.75, so this is a STRONG baseline.
#   3. OLS + spatial lag — own features PLUS the k-NN mean of each
#                        feature. This is the critical one: it gets
#                        neighbourhood context with no GNN at all.
#                        If the GNN cannot beat this, message passing
#                        is not earning its place.
#   4. random forest   — nonlinearity without spatial structure.
#   5. GNN             — 2-layer attention over the k-NN graph.
#
# VALIDATION: spatially blocked, not random. Neighbouring nodes are
# strongly autocorrelated, so a random split puts near-duplicates on
# both sides and every model scores inflated. The region is cut into
# a grid of blocks and whole blocks are held out.
#
# The GNN sees held-out nodes' FEATURES during message passing but
# never their targets (loss is masked to training nodes). That is
# standard transductive node regression; it is also what a
# cross-city transfer run would not have, so treat the transfer
# result as the stricter test when you get there.
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv
from sklearn.ensemble import RandomForestRegressor

FEATS = ['NDBI', 'NDVI', 'bld_cover', 'bld_height', 'traffic_no2_proxy']
TARGET = 'LST_landsat'
N_BLOCKS = 5          # 5x5 grid -> 25 blocks
N_FOLDS = 5
K = 8
EPOCHS = 300
SEED = 10


class LSTGNN(nn.Module):
    """Two attention layers over the k-NN graph, then a linear head.
    No LSTM, no dual head -- the target is static, so there is no
    time axis and no physics/residual split to make."""
    def __init__(self, n_feat, hidden=32, heads=4, dropout=0.2):
        super().__init__()
        self.g1 = GATConv(n_feat, hidden, heads=heads, concat=True, dropout=dropout)
        self.g2 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
                                  nn.Linear(hidden, 1))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, ei):
        h = torch.nn.functional.leaky_relu(self.g1(x, ei), 0.2)
        h = self.drop(h)
        h = torch.nn.functional.leaky_relu(self.g2(h, ei), 0.2)
        return self.head(h).squeeze(-1)


def ols_fit_predict(Xtr, ytr, Xte):
    A = np.column_stack([np.ones(len(Xtr)), Xtr])
    beta = np.linalg.lstsq(A, ytr, rcond=None)[0]
    return np.column_stack([np.ones(len(Xte)), Xte]) @ beta


def r2_score_(y, p):
    return 1 - float(((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def rmse_(y, p):
    return float(np.sqrt(((y - p) ** 2).mean()))


def run_config(cfg):
    city, month = cfg['city'], cfg['month']
    D = prepare(cfg, HP)
    gdf, node_ids = D['gdf'], D['gdf'][ID_COL].values

    raw = pd.read_csv(cfg['csv'])
    raw[ID_COL] = _align_ids(raw[ID_COL])
    tab = raw.groupby(ID_COL)[FEATS + [TARGET]].mean().reindex(node_ids)
    good = tab.notna().all(axis=1).values

    X = tab[FEATS].values[good].astype(np.float64)
    y = tab[TARGET].values[good].astype(np.float64)
    pos = np.column_stack([gdf.geometry.centroid.x.values,
                           gdf.geometry.centroid.y.values])[good]

    # spatial blocks
    bx = np.floor((pos[:, 0] - pos[:, 0].min()) /
                  (np.ptp(pos[:, 0]) + 1e-9) * N_BLOCKS).clip(0, N_BLOCKS - 1)
    by = np.floor((pos[:, 1] - pos[:, 1].min()) /
                  (np.ptp(pos[:, 1]) + 1e-9) * N_BLOCKS).clip(0, N_BLOCKS - 1)
    block = (bx * N_BLOCKS + by).astype(int)
    ublocks = np.unique(block)
    rng = np.random.default_rng(SEED)
    folds = np.array_split(rng.permutation(ublocks), N_FOLDS)

    # graph over the surviving nodes
    pos_t = torch.tensor(pos, dtype=torch.float, device=device)
    ei = knn_graph(pos_t, k=K, loop=False).to(device)
    nbr = ei[0].cpu().numpy().reshape(-1, K)      # each row: node's neighbours

    rows = []
    for fi, held in enumerate(folds):
        te = np.isin(block, held)
        tr = ~te
        if te.sum() < 30 or tr.sum() < 100:
            continue

        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
        Xz = (X - mu) / sd
        ym, ys = y[tr].mean(), y[tr].std() + 1e-9

        # spatial lag features: k-NN mean of each standardised feature
        Xlag = Xz[nbr].mean(axis=1)
        Xsp = np.column_stack([Xz, Xlag])

        preds = {
            'mean': np.full(te.sum(), y[tr].mean()),
            'OLS': ols_fit_predict(Xz[tr], y[tr], Xz[te]),
            'OLS+spatial lag': ols_fit_predict(Xsp[tr], y[tr], Xsp[te]),
        }

        rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                   random_state=SEED, n_jobs=-1)
        rf.fit(Xz[tr], y[tr])
        preds['random forest'] = rf.predict(Xz[te])

        # ---- GNN ----
        set_seed(SEED)
        xt = torch.tensor(Xz, dtype=torch.float, device=device)
        yt = torch.tensor((y - ym) / ys, dtype=torch.float, device=device)
        trm = torch.tensor(tr, device=device)
        tem = torch.tensor(te, device=device)

        net = LSTGNN(len(FEATS)).to(device)
        opt = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-4)
        best, best_state = float('inf'), None
        for ep in range(EPOCHS):
            net.train(); opt.zero_grad()
            loss = ((net(xt, ei)[trm] - yt[trm]) ** 2).mean()
            loss.backward(); opt.step()
            if ep % 10 == 0:
                net.eval()
                with torch.no_grad():
                    l = float(((net(xt, ei)[trm] - yt[trm]) ** 2).mean())
                if l < best:
                    best = l
                    best_state = {k: v.detach().clone()
                                  for k, v in net.state_dict().items()}
        if best_state:
            net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            preds['GNN'] = (net(xt, ei)[tem].cpu().numpy() * ys + ym)

        for name, p in preds.items():
            rows.append(dict(city=city, month=month, fold=fi, model=name,
                             r2=r2_score_(y[te], p), rmse=rmse_(y[te], p),
                             n_test=int(te.sum())))
    return pd.DataFrame(rows)


all_res = []
for cfg in CONFIGS:
    print(f"\n### {cfg['city']} — {cfg['month']}")
    r = run_config(cfg)
    all_res.append(r)
    piv = r.groupby('model')[['r2', 'rmse']].agg(['mean', 'std'])
    print(piv.round(4).to_string())

res = pd.concat(all_res, ignore_index=True)
res.to_csv(f"{OUT_ROOT}/gnn_vs_regression.csv", index=False)

print("\n" + "=" * 78)
print("POOLED ACROSS ALL CONFIGS AND FOLDS  (spatially blocked hold-out)")
print("=" * 78)
order = ['mean', 'OLS', 'OLS+spatial lag', 'random forest', 'GNN']
summ = (res.groupby('model')[['r2', 'rmse']]
        .agg(['mean', 'std']).reindex(order))
print(summ.round(4).to_string())

g = res[res.model == 'GNN'].groupby(['city', 'month', 'fold']).r2.mean()
s = res[res.model == 'OLS+spatial lag'].groupby(['city', 'month', 'fold']).r2.mean()
o = res[res.model == 'OLS'].groupby(['city', 'month', 'fold']).r2.mean()
d_sp, d_ols = (g - s), (g - o)
print(f"""
per-fold R2 difference, GNN minus baseline (n={len(d_sp)} folds):
  vs OLS              mean {d_ols.mean():+.4f}   std {d_ols.std():.4f}
  wins {int((d_ols > 0).sum())}/{len(d_ols)}
  vs OLS+spatial lag  mean {d_sp.mean():+.4f}   std {d_sp.std():.4f}
  wins {int((d_sp > 0).sum())}/{len(d_sp)}

The spatial-lag comparison is the one that matters. It already has
neighbourhood context, so beating plain OLS only shows that context
helps -- not that a GNN is the way to get it.

  GNN > spatial lag consistently  -> message passing learns something
      neighbour-averaging cannot. That is a result worth a paper.
  GNN ~= spatial lag              -> the gain is neighbourhood context,
      and a two-line feature engineering step captures it. Say so.
""")

In [ ]:
# ============================================================
# FAIR REMATCH — one architectural fix, one stronger baseline
# ------------------------------------------------------------
# First pass: GNN won 1/20 folds (R2 0.41 vs 0.63 for spatial-lag
# OLS). Two reasons that comparison was not yet decisive:
#
# 1. MY GNN WAS NOT YOUR GNN. LSTGNN was plain 2-layer GAT. Your
#    RobustSAGATLayer has a `self_linear` skip carrying each node's
#    own features past the aggregation, plus PairNorm. Two rounds of
#    attention over k=8 averages an effective ~64-node neighbourhood,
#    and LST has fine spatial detail -- over-smoothing is the obvious
#    mechanism for the loss, and the skip exists to prevent it.
#    Reporting "GNNs do not help" while omitting it would be testing
#    my code, not your architecture.
#
# 2. THE BEST NON-GNN MODEL HAD NOT BEEN RUN. Random forest beat
#    spatial-lag OLS in both Hyderabad configs (0.734 vs 0.683,
#    0.725 vs 0.659), so nonlinearity buys something there.
#    RF on the spatial-lag features combines the two things that each worked.
#
# BOTH were motivated before seeing their results. This is the last
# round: adding models until one wins is architecture-shopping and
# does not replicate. If the GNN still loses here, that is the answer.
#
# Also fixed: checkpoint selection now uses a held-out VALIDATION
# BLOCK from inside the training folds, not the training loss itself.
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch_geometric.nn import GATConv, PairNorm
from sklearn.ensemble import RandomForestRegressor

FEATS = ['NDBI', 'NDVI', 'bld_cover', 'bld_height', 'traffic_no2_proxy']
TARGET = 'LST_landsat'
N_BLOCKS, N_FOLDS, K = 5, 5, 8
EPOCHS, SEED = 400, 10


class SkipGATLayer(nn.Module):
    """Your RobustSAGATLayer, minus the temporal plumbing. The
    self_linear branch is the point: it lets a node's own features
    reach the output without being averaged with its neighbours."""
    def __init__(self, cin, cout, heads=4, dropout=0.2):
        super().__init__()
        self.gat = GATConv(cin, cout, heads=heads, concat=False,
                           add_self_loops=False, dropout=dropout)
        self.self_linear = nn.Linear(cin, cout)
        self.fusion = nn.Linear(cout * 2, cout)
        self.norm = PairNorm(scale=1.0)

    def forward(self, x, ei):
        nb = self.gat(x, ei)
        sf = self.self_linear(x)
        out = Fn.leaky_relu(self.fusion(torch.cat([sf, nb], -1)), 0.2)
        return self.norm(out + sf)


class SkipGNN(nn.Module):
    def __init__(self, n_feat, hidden=32, dropout=0.2):
        super().__init__()
        self.l1 = SkipGATLayer(n_feat, hidden, dropout=dropout)
        self.l2 = SkipGATLayer(hidden, hidden, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
                                  nn.Linear(hidden, 1))

    def forward(self, x, ei):
        h = self.drop(self.l1(x, ei))
        h = self.l2(h, ei)
        return self.head(h).squeeze(-1)


def ols(Xtr, ytr, Xte):
    A = np.column_stack([np.ones(len(Xtr)), Xtr])
    b = np.linalg.lstsq(A, ytr, rcond=None)[0]
    return np.column_stack([np.ones(len(Xte)), Xte]) @ b


def r2_(y, p): return 1 - float(((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())
def rmse_(y, p): return float(np.sqrt(((y - p) ** 2).mean()))


def run_config(cfg):
    city, month = cfg['city'], cfg['month']
    D = prepare(cfg, HP)
    gdf, node_ids = D['gdf'], D['gdf'][ID_COL].values
    raw = pd.read_csv(cfg['csv']); raw[ID_COL] = _align_ids(raw[ID_COL])
    tab = raw.groupby(ID_COL)[FEATS + [TARGET]].mean().reindex(node_ids)
    good = tab.notna().all(axis=1).values

    X = tab[FEATS].values[good].astype(np.float64)
    y = tab[TARGET].values[good].astype(np.float64)
    pos = np.column_stack([gdf.geometry.centroid.x.values,
                           gdf.geometry.centroid.y.values])[good]

    bx = np.floor((pos[:, 0] - pos[:, 0].min()) / (np.ptp(pos[:, 0]) + 1e-9)
                  * N_BLOCKS).clip(0, N_BLOCKS - 1)
    by = np.floor((pos[:, 1] - pos[:, 1].min()) / (np.ptp(pos[:, 1]) + 1e-9)
                  * N_BLOCKS).clip(0, N_BLOCKS - 1)
    block = (bx * N_BLOCKS + by).astype(int)
    rng = np.random.default_rng(SEED)
    folds = np.array_split(rng.permutation(np.unique(block)), N_FOLDS)

    pos_t = torch.tensor(pos, dtype=torch.float, device=device)
    ei = knn_graph(pos_t, k=K, loop=False).to(device)
    nbr = ei[0].cpu().numpy().reshape(-1, K)

    rows = []
    for fi, held in enumerate(folds):
        te = np.isin(block, held); tr = ~te
        if te.sum() < 30 or tr.sum() < 100:
            continue

        # validation blocks carved out of the TRAINING blocks, so
        # checkpoint selection never touches the test fold
        tr_blocks = np.setdiff1d(np.unique(block), held)
        val_blocks = rng.choice(tr_blocks, max(1, len(tr_blocks) // 5),
                                replace=False)
        va = np.isin(block, val_blocks) & tr
        trf = tr & ~va

        mu, sd = X[trf].mean(0), X[trf].std(0) + 1e-9
        Xz = (X - mu) / sd
        ym, ys = y[trf].mean(), y[trf].std() + 1e-9
        Xsp = np.column_stack([Xz, Xz[nbr].mean(axis=1)])

        preds = {
            'OLS': ols(Xz[trf], y[trf], Xz[te]),
            'OLS+spatial lag': ols(Xsp[trf], y[trf], Xsp[te]),
        }
        for name, M in (('random forest', Xz), ('RF+spatial lag', Xsp)):
            rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                       random_state=SEED, n_jobs=-1)
            rf.fit(M[trf], y[trf])
            preds[name] = rf.predict(M[te])

        set_seed(SEED)
        xt = torch.tensor(Xz, dtype=torch.float, device=device)
        yt = torch.tensor((y - ym) / ys, dtype=torch.float, device=device)
        trm = torch.tensor(trf, device=device)
        vam = torch.tensor(va, device=device)
        tem = torch.tensor(te, device=device)

        net = SkipGNN(len(FEATS)).to(device)
        opt = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-4)
        best, best_state = float('inf'), None
        for ep in range(EPOCHS):
            net.train(); opt.zero_grad()
            (((net(xt, ei)[trm] - yt[trm]) ** 2).mean()).backward()
            opt.step()
            if ep % 10 == 0:
                net.eval()
                with torch.no_grad():
                    v = float(((net(xt, ei)[vam] - yt[vam]) ** 2).mean())
                if v < best:
                    best = v
                    best_state = {k: t.detach().clone()
                                  for k, t in net.state_dict().items()}
        if best_state:
            net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            preds['GNN (skip)'] = net(xt, ei)[tem].cpu().numpy() * ys + ym

        for name, p in preds.items():
            rows.append(dict(city=city, month=month, fold=fi, model=name,
                             r2=r2_(y[te], p), rmse=rmse_(y[te], p)))
    return pd.DataFrame(rows)


res = pd.concat([run_config(c) for c in CONFIGS], ignore_index=True)
res.to_csv(f"{OUT_ROOT}/gnn_rematch.csv", index=False)

order = ['OLS', 'OLS+spatial lag', 'random forest', 'RF+spatial lag', 'GNN (skip)']
print("\n" + "=" * 78)
print("POOLED — spatially blocked hold-out, 20 folds")
print("=" * 78)
print(res.groupby('model')[['r2', 'rmse']].agg(['mean', 'std'])
      .reindex(order).round(4).to_string())

print("\nper config:")
for (c, m), grp in res.groupby(['city', 'month']):
    best = grp.groupby('model').r2.mean().idxmax()
    print(f"  {c} {m}: best = {best}")
    print("    " + "  ".join(f"{k}={v:.3f}"
          for k, v in grp.groupby('model').r2.mean().reindex(order).items()))

key = res.pivot_table(index=['city', 'month', 'fold'], columns='model', values='r2')
for base in ['OLS+spatial lag', 'RF+spatial lag']:
    d = key['GNN (skip)'] - key[base]
    print(f"\nGNN (skip) minus {base}:  mean {d.mean():+.4f}  "
          f"std {d.std():.4f}  wins {int((d > 0).sum())}/{len(d)}")

print("""
If the GNN now wins consistently, the skip connection was the missing
piece and the finding is about architecture, not about graphs.
If it still loses, that is the result: on this task, at this node
spacing, spatial context helps but message passing is not how to get
it -- and the strongest model is a forest on lag features.
""")


In [ ]:
# ============================================================
# COMPONENT ABLATION — what actually caused 0.41 -> 0.68?
# ------------------------------------------------------------
# The two earlier scripts differed in SIX ways at once, so the gap
# cannot be attributed yet:
#   1. skip connection (self_linear + fusion)
#   2. PairNorm
#   3. add_self_loops True vs False
#   4. 300 vs 400 epochs
#   5. checkpoint selection on TRAINING loss vs held-out VALIDATION
#      blocks   <- on its own capable of moving R2 a lot
#   6. hidden/head layout
#
# Everything below uses ONE fixed training protocol. One component
# moves at a time.
#
# STAGE 1 — MECHANISM (the important one).
#   If over-smoothing is really the cause, plain GAT must DEGRADE
#   with depth while the skip variant does not. A depth sweep is
#   direct evidence for the mechanism; "skip helps" alone is only
#   evidence that something helps. This is what turns the finding
#   from an observation into an explanation, and it is what a
#   reviewer will want.
#
# STAGE 2 — HEADLINE, with 3 seeds.
#   The claimed margin over spatial-lag OLS was +0.05 with fold std
#   0.086. One seed cannot separate that from initialisation luck.
#
# STAGE 3 — PROTOCOL CONTROL.
#   Same architecture, checkpoint chosen on training loss instead of
#   validation blocks. Isolates how much of the original gap was my
#   protocol change rather than the architecture.
#
# Runtime: roughly 300-350 short fits. Early stopping keeps most of
# them well under the epoch cap. Set CONFIGS_USED to a subset while
# testing.
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch_geometric.nn import GATConv, PairNorm
from sklearn.ensemble import RandomForestRegressor

FEATS = ['NDBI', 'NDVI', 'bld_cover', 'bld_height', 'traffic_no2_proxy']
TARGET = 'LST_landsat'
N_BLOCKS, N_FOLDS, K = 5, 5, 8
EPOCHS, PATIENCE, HIDDEN, HEADS = 400, 60, 32, 4
SEEDS = [10, 11, 12]
CONFIGS_USED = CONFIGS          # subset while testing


class Layer(nn.Module):
    def __init__(self, cin, cout, use_skip, use_pairnorm, self_loops,
                 heads=HEADS, dropout=0.2):
        super().__init__()
        self.use_skip, self.use_pairnorm = use_skip, use_pairnorm
        self.gat = GATConv(cin, cout, heads=heads, concat=False,
                           add_self_loops=self_loops, dropout=dropout)
        if use_skip:
            self.self_linear = nn.Linear(cin, cout)
            self.fusion = nn.Linear(cout * 2, cout)
        if use_pairnorm:
            self.norm = PairNorm(scale=1.0)

    def forward(self, x, ei):
        nb = self.gat(x, ei)
        if self.use_skip:
            sf = self.self_linear(x)
            out = Fn.leaky_relu(self.fusion(torch.cat([sf, nb], -1)), 0.2) + sf
        else:
            out = Fn.leaky_relu(nb, 0.2)
        return self.norm(out) if self.use_pairnorm else out


class Net(nn.Module):
    def __init__(self, n_feat, depth, use_skip, use_pairnorm, self_loops,
                 hidden=HIDDEN, dropout=0.2):
        super().__init__()
        dims = [n_feat] + [hidden] * depth
        self.layers = nn.ModuleList([
            Layer(dims[i], dims[i + 1], use_skip, use_pairnorm, self_loops)
            for i in range(depth)])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
                                  nn.Linear(hidden, 1))

    def forward(self, x, ei):
        for i, l in enumerate(self.layers):
            x = l(x, ei)
            if i < len(self.layers) - 1:
                x = self.drop(x)
        return self.head(x).squeeze(-1)


def r2_(y, p): return 1 - float(((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def ols(Xtr, ytr, Xte):
    A = np.column_stack([np.ones(len(Xtr)), Xtr])
    b = np.linalg.lstsq(A, ytr, rcond=None)[0]
    return np.column_stack([np.ones(len(Xte)), Xte]) @ b


def build(cfg):
    """Node table, graph and spatial blocks for one config."""
    D = prepare(cfg, HP)
    gdf, node_ids = D['gdf'], D['gdf'][ID_COL].values
    raw = pd.read_csv(cfg['csv']); raw[ID_COL] = _align_ids(raw[ID_COL])
    tab = raw.groupby(ID_COL)[FEATS + [TARGET]].mean().reindex(node_ids)
    good = tab.notna().all(axis=1).values

    X = tab[FEATS].values[good].astype(np.float64)
    y = tab[TARGET].values[good].astype(np.float64)
    pos = np.column_stack([gdf.geometry.centroid.x.values,
                           gdf.geometry.centroid.y.values])[good]

    bx = np.floor((pos[:, 0] - pos[:, 0].min()) / (np.ptp(pos[:, 0]) + 1e-9)
                  * N_BLOCKS).clip(0, N_BLOCKS - 1)
    by = np.floor((pos[:, 1] - pos[:, 1].min()) / (np.ptp(pos[:, 1]) + 1e-9)
                  * N_BLOCKS).clip(0, N_BLOCKS - 1)
    block = (bx * N_BLOCKS + by).astype(int)

    ei = knn_graph(torch.tensor(pos, dtype=torch.float, device=device),
                   k=K, loop=False).to(device)
    nbr = ei[0].cpu().numpy().reshape(-1, K)
    return X, y, block, ei, nbr


def fit_gnn(X, y, ei, trf, va, te, depth, use_skip, use_pairnorm,
            self_loops, seed, select_on='val'):
    mu, sd = X[trf].mean(0), X[trf].std(0) + 1e-9
    Xz = (X - mu) / sd
    ym, ys = y[trf].mean(), y[trf].std() + 1e-9

    set_seed(seed)
    xt = torch.tensor(Xz, dtype=torch.float, device=device)
    yt = torch.tensor((y - ym) / ys, dtype=torch.float, device=device)
    trm = torch.tensor(trf, device=device)
    vam = torch.tensor(va, device=device)
    tem = torch.tensor(te, device=device)

    net = Net(X.shape[1], depth, use_skip, use_pairnorm, self_loops).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-4)
    best, best_state, since = float('inf'), None, 0
    for ep in range(EPOCHS):
        net.train(); opt.zero_grad()
        loss = ((net(xt, ei)[trm] - yt[trm]) ** 2).mean()
        loss.backward(); opt.step()
        if ep % 10 == 0:
            net.eval()
            with torch.no_grad():
                crit = (float(((net(xt, ei)[vam] - yt[vam]) ** 2).mean())
                        if select_on == 'val' else float(loss))
            if crit < best - 1e-5:
                best, since = crit, 0
                best_state = {k: t.detach().clone()
                              for k, t in net.state_dict().items()}
            else:
                since += 10
                if since >= PATIENCE:
                    break
    if best_state:
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        return net(xt, ei)[tem].cpu().numpy() * ys + ym


# ---- arms -------------------------------------------------------
# (name, depth, skip, pairnorm, self_loops, select_on)
STAGE1 = [(f"plain d{d}", d, False, False, True, 'val') for d in (1, 2, 3)] + \
         [(f"skip  d{d}", d, True, False, False, 'val') for d in (1, 2, 3)]
STAGE2 = [("plain d2 +PN", 2, False, True, True, 'val'),
          ("skip  d2 +PN", 2, True, True, False, 'val')]
STAGE3 = [("skip d2 +PN, train-sel", 2, True, True, False, 'train')]

rows = []
for cfg in CONFIGS_USED:
    city, month = cfg['city'], cfg['month']
    print(f"\n### {city} {month}")
    X, y, block, ei, nbr = build(cfg)
    rng = np.random.default_rng(10)
    folds = np.array_split(rng.permutation(np.unique(block)), N_FOLDS)

    for fi, held in enumerate(folds):
        te = np.isin(block, held); tr = ~te
        if te.sum() < 30 or tr.sum() < 100:
            continue
        tr_blocks = np.setdiff1d(np.unique(block), held)
        val_blocks = rng.choice(tr_blocks, max(1, len(tr_blocks) // 5),
                                replace=False)
        va = np.isin(block, val_blocks) & tr
        trf = tr & ~va

        # non-GNN references, once per fold
        mu, sd = X[trf].mean(0), X[trf].std(0) + 1e-9
        Xz = (X - mu) / sd
        Xsp = np.column_stack([Xz, Xz[nbr].mean(axis=1)])
        for nm, p in (('OLS', ols(Xz[trf], y[trf], Xz[te])),
                      ('OLS+spatial lag', ols(Xsp[trf], y[trf], Xsp[te]))):
            rows.append(dict(city=city, month=month, fold=fi, seed=-1,
                             model=nm, r2=r2_(y[te], p)))
        rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                   random_state=10, n_jobs=-1).fit(Xsp[trf], y[trf])
        rows.append(dict(city=city, month=month, fold=fi, seed=-1,
                         model='RF+spatial lag', r2=r2_(y[te], rf.predict(Xsp[te]))))

        # stage 1 + 3: one seed
        for nm, d, sk, pn, sl, sel in STAGE1 + STAGE3:
            p = fit_gnn(X, y, ei, trf, va, te, d, sk, pn, sl, SEEDS[0], sel)
            rows.append(dict(city=city, month=month, fold=fi, seed=SEEDS[0],
                             model=nm, r2=r2_(y[te], p)))
        # stage 2: three seeds
        for nm, d, sk, pn, sl, sel in STAGE2:
            for s in SEEDS:
                p = fit_gnn(X, y, ei, trf, va, te, d, sk, pn, sl, s, sel)
                rows.append(dict(city=city, month=month, fold=fi, seed=s,
                                 model=nm, r2=r2_(y[te], p)))
    print(f"  folds done")

res = pd.DataFrame(rows)
res.to_csv(f"{OUT_ROOT}/component_ablation.csv", index=False)

print("\n" + "=" * 74)
print("STAGE 1 — DEPTH SWEEP (the mechanism test)")
print("=" * 74)
d1 = res[res.model.str.contains('plain d|skip  d')]
print(d1.groupby('model').r2.agg(['mean', 'std']).round(4).to_string())
print("""
  plain DEGRADES with depth, skip does not -> over-smoothing confirmed
      as the mechanism. This is the figure the paper is built on.
  both flat across depth -> the gap is not over-smoothing; whatever
      the skip does, do not explain it that way.
  plain never degrades -> at k=8 and 500 m, 3 layers may be too
      shallow to over-smooth. Extend to depth 4-6 before concluding.
""")

print("=" * 74)
print("STAGE 2 — HEADLINE, 3 seeds")
print("=" * 74)
d2 = res[res.model.str.contains(r'\+PN$') | (res.seed == -1)]
print(d2.groupby('model').r2.agg(['mean', 'std', 'count']).round(4).to_string())

key = (res[res.model.isin(['skip  d2 +PN', 'OLS+spatial lag', 'RF+spatial lag'])]
       .groupby(['city', 'month', 'fold', 'model']).r2.mean().unstack())
for base in ['OLS+spatial lag', 'RF+spatial lag']:
    if 'skip  d2 +PN' in key and base in key:
        dd = (key['skip  d2 +PN'] - key[base]).dropna()
        print(f"\n  skip d2 +PN minus {base}: mean {dd.mean():+.4f}  "
              f"std {dd.std():.4f}  wins {int((dd > 0).sum())}/{len(dd)}")

seed_sd = (res[res.model == 'skip  d2 +PN']
           .groupby(['city', 'month', 'fold']).r2.std().mean())
print(f"\n  mean within-fold SEED std for skip d2 +PN: {seed_sd:.4f}")
print("  Compare that against the margins above. If the margin is not")
print("  clearly larger than the seed spread, it is not a result.")

print("\n" + "=" * 74)
print("STAGE 3 — PROTOCOL CONTROL")
print("=" * 74)
a = res[res.model == 'skip  d2 +PN'].groupby(['city', 'month', 'fold']).r2.mean()
b = res[res.model == 'skip d2 +PN, train-sel'].groupby(['city', 'month', 'fold']).r2.mean()
dd = (a - b).dropna()
print(f"  validation-block selection minus training-loss selection:")
print(f"    mean {dd.mean():+.4f}   std {dd.std():.4f}   "
      f"helps in {int((dd > 0).sum())}/{len(dd)} folds")
print("""
  If this is large, a good share of the original 0.41 -> 0.68 gap was
  the protocol change, not the skip connection -- and the finding must
  be stated that way. If it is small, the architecture claim holds.
""")

In [ ]:
# ============================================================
# PART A: DEPTH SWEEP 1-6   |   PART B: CROSS-CITY TRANSFER
# ------------------------------------------------------------
# PART A extends the mechanism test. At depths 1-3 plain GAT fell
# 0.492 -> 0.420 -> 0.343 while skip stayed flat (0.682/0.690/0.684).
# Three points is a trend; six is a figure. If plain keeps falling and
# skip stays flat to depth 6, the over-smoothing account is solid.
#
# Note the gap already exists at DEPTH 1 (0.492 vs 0.682). One hop
# cannot over-smooth, so two effects are in play: a depth-INDEPENDENT
# penalty (a node's own features diluted among 8 neighbours, which the
# skip preserves) and a depth-DEPENDENT one (over-smoothing). Part A
# separates them: the depth-1 gap measures the first, the slope
# measures the second. Say both in the paper.
#
# PART B is the strongest claim still available: train on one city,
# predict the other, no retraining and no fine-tuning. Nothing about
# the target city's graph, features or temperatures is used.
#
# Two normalisation regimes, because the distinction matters:
#   STRICT   — standardise target features with SOURCE statistics.
#              True zero-shot; no target information at all.
#   ADAPTED  — standardise with TARGET feature statistics (features
#              only, never the target variable). Standard practice in
#              domain adaptation and defensible, but it is not
#              zero-shot and must not be reported as such.
#
# Two metrics, because absolute transfer and pattern transfer are
# different claims:
#   R2 absolute — includes any mean offset between cities. Harsh: a
#                 constant bias tanks it even if the pattern is right.
#   R2 anomaly  — after removing each city's mean LST. This is the
#                 question a heat MAP actually poses: does the spatial
#                 pattern transfer?
# ============================================================
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestRegressor

DEPTHS = [1, 2, 3, 4, 5, 6]
TRANSFER_PAIRS = [(0, 2), (2, 0), (1, 3), (3, 1)]   # indices into CONFIGS
SEEDS_A, SEEDS_B = [10], [10, 11, 12]

# reuse Layer / Net / fit_gnn / build / r2_ / ols from the ablation cell
cache = {c['city'] + c['month']: build(c) for c in CONFIGS}

# ============================================================
# PART A — depth sweep
# ============================================================
rows = []
for cfg in CONFIGS:
    city, month = cfg['city'], cfg['month']
    X, y, block, ei, nbr = cache[city + month]
    rng = np.random.default_rng(10)
    folds = np.array_split(rng.permutation(np.unique(block)), N_FOLDS)
    print(f"### depth sweep — {city} {month}")

    for fi, held in enumerate(folds):
        te = np.isin(block, held); tr = ~te
        if te.sum() < 30 or tr.sum() < 100:
            continue
        tr_blocks = np.setdiff1d(np.unique(block), held)
        va = np.isin(block, rng.choice(tr_blocks, max(1, len(tr_blocks) // 5),
                                       replace=False)) & tr
        trf = tr & ~va
        for d in DEPTHS:
            for nm, sk, sl in (('plain', False, True), ('skip', True, False)):
                p = fit_gnn(X, y, ei, trf, va, te, d, sk, False, sl,
                            SEEDS_A[0], 'val')
                rows.append(dict(city=city, month=month, fold=fi,
                                 arch=nm, depth=d, r2=r2_(y[te], p)))

dep = pd.DataFrame(rows)
dep.to_csv(f"{OUT_ROOT}/depth_sweep.csv", index=False)

print("\n" + "=" * 70)
print("PART A — R2 vs DEPTH")
print("=" * 70)
piv = dep.pivot_table(index='depth', columns='arch', values='r2',
                      aggfunc=['mean', 'std'])
print(piv.round(4).to_string())

pm = dep[dep.arch == 'plain'].groupby('depth').r2.mean()
sm = dep[dep.arch == 'skip'].groupby('depth').r2.mean()
print(f"\n  plain: depth 1 = {pm.iloc[0]:.3f} -> depth {DEPTHS[-1]} = "
      f"{pm.iloc[-1]:.3f}   slope {np.polyfit(DEPTHS, pm.values, 1)[0]:+.4f}/layer")
print(f"  skip : depth 1 = {sm.iloc[0]:.3f} -> depth {DEPTHS[-1]} = "
      f"{sm.iloc[-1]:.3f}   slope {np.polyfit(DEPTHS, sm.values, 1)[0]:+.4f}/layer")
print(f"\n  depth-1 gap (feature dilution):     {sm.iloc[0] - pm.iloc[0]:+.3f}")
print(f"  extra gap by depth {DEPTHS[-1]} (over-smoothing): "
      f"{(sm.iloc[-1] - pm.iloc[-1]) - (sm.iloc[0] - pm.iloc[0]):+.3f}")

# ============================================================
# PART B — cross-city transfer
# ============================================================
def transfer(src_cfg, dst_cfg, regime):
    """Fit on source, predict target. No target labels are ever used."""
    sX, sy, sblock, sei, snbr = cache[src_cfg['city'] + src_cfg['month']]
    tX, ty, tblock, tei, tnbr = cache[dst_cfg['city'] + dst_cfg['month']]

    mu, sd = sX.mean(0), sX.std(0) + 1e-9
    tmu, tsd = (mu, sd) if regime == 'strict' else (tX.mean(0), tX.std(0) + 1e-9)
    sXz, tXz = (sX - mu) / sd, (tX - tmu) / tsd
    sXsp = np.column_stack([sXz, sXz[snbr].mean(axis=1)])
    tXsp = np.column_stack([tXz, tXz[tnbr].mean(axis=1)])
    ym, ys = sy.mean(), sy.std() + 1e-9

    out = {'OLS': ols(sXz, sy, tXz),
           'OLS+spatial lag': ols(sXsp, sy, tXsp)}
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                               random_state=10, n_jobs=-1).fit(sXsp, sy)
    out['RF+spatial lag'] = rf.predict(tXsp)

    # validation blocks inside the SOURCE city drive early stopping
    rng = np.random.default_rng(10)
    vb = rng.choice(np.unique(sblock), max(1, len(np.unique(sblock)) // 5),
                    replace=False)
    va = np.isin(sblock, vb)
    trf = ~va

    for nm, d, sk, sl in (('plain d2', 2, False, True),
                          ('skip d2', 2, True, False)):
        preds = []
        for s in SEEDS_B:
            mu2, sd2 = sX[trf].mean(0), sX[trf].std(0) + 1e-9
            set_seed(s)
            net = Net(sX.shape[1], d, sk, False, sl).to(device)
            optm = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-4)
            xs = torch.tensor((sX - mu2) / sd2, dtype=torch.float, device=device)
            ysn = torch.tensor((sy - sy[trf].mean()) / (sy[trf].std() + 1e-9),
                               dtype=torch.float, device=device)
            trm = torch.tensor(trf, device=device)
            vam = torch.tensor(va, device=device)
            best, bstate, since = float('inf'), None, 0
            for ep in range(EPOCHS):
                net.train(); optm.zero_grad()
                ((net(xs, sei)[trm] - ysn[trm]) ** 2).mean().backward()
                optm.step()
                if ep % 10 == 0:
                    net.eval()
                    with torch.no_grad():
                        v = float(((net(xs, sei)[vam] - ysn[vam]) ** 2).mean())
                    if v < best - 1e-5:
                        best, since = v, 0
                        bstate = {k: t.detach().clone()
                                  for k, t in net.state_dict().items()}
                    else:
                        since += 10
                        if since >= PATIENCE:
                            break
            if bstate:
                net.load_state_dict(bstate)
            net.eval()
            # target features standardised per the regime, then the
            # SOURCE model applied to the TARGET graph
            tmu2, tsd2 = (mu2, sd2) if regime == 'strict' else (tX.mean(0), tX.std(0) + 1e-9)
            xt = torch.tensor((tX - tmu2) / tsd2, dtype=torch.float, device=device)
            with torch.no_grad():
                preds.append(net(xt, tei).cpu().numpy()
                             * (sy[trf].std() + 1e-9) + sy[trf].mean())
        out[nm] = np.mean(preds, axis=0)

    res = []
    for nm, p in out.items():
        res.append(dict(
            source=f"{src_cfg['short']} {src_cfg['month']}",
            target=f"{dst_cfg['short']} {dst_cfg['month']}",
            regime=regime, model=nm,
            r2_absolute=r2_(ty, p),
            r2_anomaly=r2_(ty - ty.mean(), p - p.mean()),
            bias=float(p.mean() - ty.mean())))
    return res


trows = []
for si, di in TRANSFER_PAIRS:
    for regime in ('strict', 'adapted'):
        print(f"### transfer {CONFIGS[si]['short']} -> {CONFIGS[di]['short']} "
              f"({CONFIGS[si]['month']}, {regime})")
        trows += transfer(CONFIGS[si], CONFIGS[di], regime)

tr = pd.DataFrame(trows)
tr.to_csv(f"{OUT_ROOT}/cross_city_transfer.csv", index=False)

print("\n" + "=" * 78)
print("PART B — ZERO-SHOT CROSS-CITY TRANSFER")
print("=" * 78)
for regime in ('strict', 'adapted'):
    print(f"\n--- {regime} normalisation ---")
    sub = tr[tr.regime == regime]
    print(sub.pivot_table(index=['source', 'target'], columns='model',
                          values='r2_anomaly').round(3).to_string())
    print("  (R2 on the spatial ANOMALY -- does the pattern transfer)")
    print(sub.groupby('model')[['r2_absolute', 'r2_anomaly', 'bias']]
          .mean().round(3).to_string())

print("""
READING PART B
  skip d2 holding up where the baselines fall  -> the graph learns
      something that generalises across cities. That is the strongest
      claim available and it is a headline result.
  everything transfers similarly  -> the relationship is largely
      shared; report it as evidence the features generalise, not the
      architecture.
  everything collapses  -> the models are city-specific. Say so; it
      bounds the claim honestly and is worth a paragraph.

  Compare STRICT against ADAPTED. A large difference means the gain
  comes from target feature statistics, not transferred structure --
  in which case only the strict numbers may be called zero-shot.
""")

In [ ]:
# ============================================================
# PART 1: SPLIT PROTOCOL   |   PART 2: OVER-SMOOTHING REMEDIES
# ------------------------------------------------------------
# PART 1 — the critique, and probably the paper's most citable piece.
#   Published GNN work on urban LST commonly splits nodes at random.
#   Under spatial autocorrelation a held-out node's k nearest
#   neighbours sit in the training set, so the model is scored on
#   near-duplicates of what it fit. This runs BOTH protocols on
#   identical data and measures the inflation. The question is not
#   only how large it is, but whether the GNN inflates MORE than the
#   regression baselines -- if it does, published GNN-vs-baseline
#   margins are partly an artifact of the split.
#
# PART 2 — the priority problem.
#   "Skip connections fix over-smoothing" is established: JKNet
#   (Xu 2018), DeepGCNs (Li 2019), APPNP (Klicpera 2019), GCNII
#   (Chen 2020), DropEdge. Comparing only against plain GAT invites
#   the obvious reviewer question. Comparing against the remedies
#   themselves converts a weak novelty claim into a benchmark: WHICH
#   remedy works for fine-grained spatial regression, and at what
#   depth. A benchmark cannot be scooped on priority.
#
# RUNTIME with six configs: ~360 GNN fits in Part 1, ~450 in Part 2,
# so 2.5-4 hours end to end. The parts toggle independently -- read
# Part 1 before committing to Part 2, since Part 1 alone may change
# how the paper is framed.
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch_geometric.nn import GATConv
from torch_geometric.utils import dropout_adj
from sklearn.ensemble import RandomForestRegressor

FEATS = ['NDBI', 'NDVI', 'bld_cover', 'bld_height', 'traffic_no2_proxy']
TARGET = 'LST_landsat'
N_BLOCKS, N_FOLDS, K = 5, 5, 8
EPOCHS, PATIENCE, HIDDEN, HEADS = 400, 60, 32, 4
SEEDS = [10, 11, 12]
REMEDY_DEPTHS = [2, 4, 6]

RUN_PART_1 = True
RUN_PART_2 = True
PROGRESS = True

# build() is defined in the component-ablation cell, not in cells 2-7.
# Fail now rather than after Part 1 has already run for an hour.
for _dep in ('build', 'set_seed', 'device', 'ID_COL', '_align_ids', 'CONFIGS'):
    if _dep not in dir():
        raise NameError(f"'{_dep}' is not defined -- run cells 2, 3, 4, 7 "
                        f"and the component-ablation cell first.")


# ============================================================
# Architectures
# ============================================================
class GATBlock(nn.Module):
    """One aggregation step, shared by every variant so the only
    difference between arms is the remedy itself."""
    def __init__(self, cin, cout, self_loops=True, dropout=0.2):
        super().__init__()
        self.gat = GATConv(cin, cout, heads=HEADS, concat=False,
                           add_self_loops=self_loops, dropout=dropout)

    def forward(self, x, ei):
        return Fn.leaky_relu(self.gat(x, ei), 0.2)


class RemedyNet(nn.Module):
    """variant:
        plain     — GAT stack, no remedy
        skip      — per-layer self_linear + fusion (yours)
        jk        — Jumping Knowledge: concatenate every layer output
        gcnii     — initial residual: h_l = (1-a)*agg(h) + a*h0
        dropedge  — plain GAT with edges randomly dropped each step
    """
    def __init__(self, n_feat, depth, variant, hidden=HIDDEN, dropout=0.2,
                 gcnii_alpha=0.1, edge_drop=0.3):
        super().__init__()
        self.variant, self.depth = variant, depth
        self.edge_drop, self.gcnii_alpha = edge_drop, gcnii_alpha
        self.drop = nn.Dropout(dropout)
        sl = (variant != 'skip')
        self.blocks, self.skips, self.fuse = (nn.ModuleList(), nn.ModuleList(),
                                              nn.ModuleList())
        dims = [n_feat] + [hidden] * depth
        for i in range(depth):
            cin = dims[i] if variant != 'gcnii' else (hidden if i else n_feat)
            self.blocks.append(GATBlock(cin, hidden, self_loops=sl, dropout=dropout))
            if variant == 'skip':
                self.skips.append(nn.Linear(cin, hidden))
                self.fuse.append(nn.Linear(hidden * 2, hidden))
        if variant == 'gcnii':
            self.h0 = nn.Linear(n_feat, hidden)
        head_in = hidden * depth if variant == 'jk' else hidden
        self.head = nn.Sequential(nn.Linear(head_in, hidden), nn.LeakyReLU(0.2),
                                  nn.Linear(hidden, 1))

    def forward(self, x, ei):
        outs, h = [], x
        h0 = self.h0(x) if self.variant == 'gcnii' else None
        for i, blk in enumerate(self.blocks):
            e = ei
            if self.variant == 'dropedge' and self.training:
                e, _ = dropout_adj(ei, p=self.edge_drop, training=True)
            nb = blk(h, e)
            if self.variant == 'skip':
                sf = self.skips[i](h)
                h = Fn.leaky_relu(self.fuse[i](torch.cat([sf, nb], -1)), 0.2) + sf
            elif self.variant == 'gcnii':
                a = self.gcnii_alpha
                h = (1 - a) * nb + a * h0
            else:
                h = nb
            outs.append(h)
            if i < self.depth - 1:
                h = self.drop(h)
        z = torch.cat(outs, -1) if self.variant == 'jk' else h
        return self.head(z).squeeze(-1)


def r2_(y, p): return 1 - float(((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def ols(Xtr, ytr, Xte):
    A = np.column_stack([np.ones(len(Xtr)), Xtr])
    b = np.linalg.lstsq(A, ytr, rcond=None)[0]
    return np.column_stack([np.ones(len(Xte)), Xte]) @ b


def fit(X, y, ei, trf, va, te, depth, variant, seed):
    mu, sd = X[trf].mean(0), X[trf].std(0) + 1e-9
    Xz = (X - mu) / sd
    ym, ys = y[trf].mean(), y[trf].std() + 1e-9
    set_seed(seed)
    xt = torch.tensor(Xz, dtype=torch.float, device=device)
    yt = torch.tensor((y - ym) / ys, dtype=torch.float, device=device)
    trm = torch.tensor(trf, device=device)
    vam = torch.tensor(va, device=device)
    tem = torch.tensor(te, device=device)
    net = RemedyNet(X.shape[1], depth, variant).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-4)
    best, bstate, since = float('inf'), None, 0
    for ep in range(EPOCHS):
        net.train(); opt.zero_grad()
        ((net(xt, ei)[trm] - yt[trm]) ** 2).mean().backward(); opt.step()
        if ep % 10 == 0:
            net.eval()
            with torch.no_grad():
                v = float(((net(xt, ei)[vam] - yt[vam]) ** 2).mean())
            if v < best - 1e-5:
                best, since = v, 0
                bstate = {k: t.detach().clone() for k, t in net.state_dict().items()}
            else:
                since += 10
                if since >= PATIENCE:
                    break
    if bstate:
        net.load_state_dict(bstate)
    net.eval()
    with torch.no_grad():
        return net(xt, ei)[tem].cpu().numpy() * ys + ym


def make_folds(block, n, protocol, rng):
    """Blocked: hold out whole spatial blocks. Random: hold out nodes
    at random, ignoring geography -- the protocol under critique."""
    if protocol == 'blocked':
        return [np.isin(block, h) for h in
                np.array_split(rng.permutation(np.unique(block)), N_FOLDS)]
    return [np.isin(np.arange(n), c)
            for c in np.array_split(rng.permutation(n), N_FOLDS)]


def val_mask(block, tr, te, protocol, rng, n):
    """Validation set carved from TRAINING data, matched to the
    protocol so early stopping never sees the test fold."""
    va = np.zeros(n, bool)
    if protocol == 'blocked':
        tb = np.setdiff1d(np.unique(block[tr]), np.unique(block[te]))
        va[np.isin(block, rng.choice(tb, max(1, len(tb) // 5),
                                     replace=False)) & tr] = True
    else:
        idx = np.where(tr)[0]
        va[rng.choice(idx, len(idx) // 5, replace=False)] = True
    return va


# ============================================================
# PART 1 — split protocol
# ============================================================
if RUN_PART_1:
    rows = []
    for cfg in CONFIGS:
        city, month = cfg['city'], cfg['month']
        X, y, block, ei, nbr = build(cfg)
        print(f"### split protocol — {city} {month}", flush=True)
        for protocol in ('blocked', 'random'):
            rng = np.random.default_rng(10)
            for fi, te in enumerate(make_folds(block, len(y), protocol, rng)):
                tr = ~te
                if te.sum() < 30 or tr.sum() < 100:
                    continue
                va = val_mask(block, tr, te, protocol, rng, len(y))
                trf = tr & ~va
                if PROGRESS:
                    print(f"    {protocol:8s} fold {fi}  "
                          f"({int(te.sum())} test nodes)", flush=True)

                mu, sd = X[trf].mean(0), X[trf].std(0) + 1e-9
                Xz = (X - mu) / sd
                Xsp = np.column_stack([Xz, Xz[nbr].mean(axis=1)])
                for nm, p in (('OLS', ols(Xz[trf], y[trf], Xz[te])),
                              ('OLS+spatial lag', ols(Xsp[trf], y[trf], Xsp[te]))):
                    rows.append(dict(city=city, month=month, protocol=protocol,
                                     fold=fi, model=nm, r2=r2_(y[te], p)))
                rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                           random_state=10, n_jobs=-1)
                rf.fit(Xsp[trf], y[trf])
                rows.append(dict(city=city, month=month, protocol=protocol,
                                 fold=fi, model='RF+spatial lag',
                                 r2=r2_(y[te], rf.predict(Xsp[te]))))
                for variant, nm in (('plain', 'GAT d2'), ('skip', 'SkipGAT d2')):
                    for s in SEEDS:
                        p = fit(X, y, ei, trf, va, te, 2, variant, s)
                        rows.append(dict(city=city, month=month, protocol=protocol,
                                         fold=fi, model=nm, seed=s,
                                         r2=r2_(y[te], p)))

    sp = pd.DataFrame(rows)
    sp.to_csv(f"{OUT_ROOT}/split_protocol.csv", index=False)

    print("\n" + "=" * 74)
    print("PART 1 — RANDOM vs SPATIALLY BLOCKED SPLITS")
    print("=" * 74)
    piv = sp.pivot_table(index='model', columns='protocol', values='r2',
                         aggfunc='mean')
    piv['inflation'] = piv['random'] - piv['blocked']
    print(piv.round(4).to_string())
    for proto in ('blocked', 'random'):
        s = piv[proto]
        print(f"\n  {proto:8s}: SkipGAT minus OLS+lag = "
              f"{s['SkipGAT d2'] - s['OLS+spatial lag']:+.4f}")
    print("""
  'inflation' is how much a random split flatters each model. If the
  GNN inflates MORE than the regression baselines, then published
  GNN-vs-baseline margins obtained under random splits are partly an
  artifact of the protocol -- which is the finding. Compare the two
  SkipGAT-minus-OLS+lag margins above: that is the headline number
  under each protocol.
""")
else:
    print("PART 1 skipped (RUN_PART_1 = False)")

# ============================================================
# PART 2 — remedy comparison, blocked validation only
# ============================================================
if RUN_PART_2:
    rows2 = []
    for cfg in CONFIGS:
        city, month = cfg['city'], cfg['month']
        X, y, block, ei, nbr = build(cfg)
        print(f"### remedies — {city} {month}", flush=True)
        rng = np.random.default_rng(10)
        for fi, te in enumerate(make_folds(block, len(y), 'blocked', rng)):
            tr = ~te
            if te.sum() < 30 or tr.sum() < 100:
                continue
            va = val_mask(block, tr, te, 'blocked', rng, len(y))
            trf = tr & ~va
            if PROGRESS:
                print(f"    fold {fi}", flush=True)
            for d in REMEDY_DEPTHS:
                for variant in ('plain', 'skip', 'jk', 'gcnii', 'dropedge'):
                    p = fit(X, y, ei, trf, va, te, d, variant, SEEDS[0])
                    rows2.append(dict(city=city, month=month, fold=fi, depth=d,
                                      variant=variant, r2=r2_(y[te], p)))

    rm = pd.DataFrame(rows2)
    rm.to_csv(f"{OUT_ROOT}/remedy_comparison.csv", index=False)

    print("\n" + "=" * 74)
    print("PART 2 — OVER-SMOOTHING REMEDIES vs DEPTH (blocked validation)")
    print("=" * 74)
    print(rm.pivot_table(index='depth', columns='variant', values='r2',
                         aggfunc='mean').round(4).to_string())
    print("\nslope per layer (degradation with depth):")
    for v in ('plain', 'skip', 'jk', 'gcnii', 'dropedge'):
        m = rm[rm.variant == v].groupby('depth').r2.mean()
        if len(m) > 1:
            print(f"  {v:10s} {np.polyfit(m.index.values, m.values, 1)[0]:+.4f}"
                  f"   depth {REMEDY_DEPTHS[0]}: {m.iloc[0]:.3f} -> "
                  f"depth {REMEDY_DEPTHS[-1]}: {m.iloc[-1]:.3f}")
    print("""
  If several remedies hold flat and yours is not clearly best, the
  honest contribution is the BENCHMARK -- which remedies work on
  fine-grained spatial regression and by how much -- not a novel
  architecture. That is the more defensible paper anyway: a benchmark
  cannot be scooped on priority, and the depth curves are the figure
  either way.
""")
else:
    print("PART 2 skipped (RUN_PART_2 = False)")

In [ ]:
# ============================================================
# CROSS-CITY TRANSFER — all three cities
# ------------------------------------------------------------
# The earlier run hardcoded TRANSFER_PAIRS as indices
# [(0,2),(2,0),(1,3),(3,1)], which only reach Bengaluru and
# Hyderabad. Pairs are now GENERATED: every ordered pair of distinct
# cities within the same season. With three cities that is 6 pairs
# per season, 12 in total, versus 4 before.
#
# Delhi is the informative addition. Bengaluru and Hyderabad are both
# southern peninsular cities; Delhi is northern, continental and
# semi-arid, so Blr<->Dlh and Hyd<->Dlh are much harder transfers than
# Blr<->Hyd. If the pattern holds there, the claim is considerably
# stronger. If it breaks only on Delhi pairs, that is itself worth
# reporting -- it would bound generalisation by climate zone rather
# than by city.
#
# Part A (depth sweep) is already complete on six configs and is NOT
# re-run here.
#
# Requires from earlier cells: build, Net, ols, r2_, set_seed, device,
# EPOCHS, PATIENCE.
# ============================================================
import numpy as np
import pandas as pd
import torch

for _dep in ('build', 'Net', 'ols', 'r2_', 'set_seed', 'EPOCHS', 'PATIENCE'):
    if _dep not in dir():
        raise NameError(f"'{_dep}' undefined -- run the component-ablation "
                        f"and depth/transfer cells first.")

SEEDS_B = [10, 11, 12]

cache = {c['city'] + c['month']: build(c) for c in CONFIGS}

# every ordered pair of DIFFERENT cities within the same season
PAIRS = [(i, j) for i, a in enumerate(CONFIGS) for j, b in enumerate(CONFIGS)
         if a['month'] == b['month'] and a['city'] != b['city']]
print(f"{len(PAIRS)} transfers: "
      + ", ".join(f"{CONFIGS[i]['short']}->{CONFIGS[j]['short']}"
                  f"({CONFIGS[i]['month'][:3]})" for i, j in PAIRS))


def transfer(src_cfg, dst_cfg, regime):
    """Fit on source, predict target. No target labels are ever used."""
    sX, sy, sblock, sei, snbr = cache[src_cfg['city'] + src_cfg['month']]
    tX, ty, tblock, tei, tnbr = cache[dst_cfg['city'] + dst_cfg['month']]

    mu, sd = sX.mean(0), sX.std(0) + 1e-9
    tmu, tsd = (mu, sd) if regime == 'strict' else (tX.mean(0), tX.std(0) + 1e-9)
    sXz, tXz = (sX - mu) / sd, (tX - tmu) / tsd
    sXsp = np.column_stack([sXz, sXz[snbr].mean(axis=1)])
    tXsp = np.column_stack([tXz, tXz[tnbr].mean(axis=1)])

    out = {'OLS': ols(sXz, sy, tXz),
           'OLS+spatial lag': ols(sXsp, sy, tXsp)}
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                               random_state=10, n_jobs=-1).fit(sXsp, sy)
    out['RF+spatial lag'] = rf.predict(tXsp)

    # validation blocks inside the SOURCE city drive early stopping
    rng = np.random.default_rng(10)
    ub = np.unique(sblock)
    va = np.isin(sblock, rng.choice(ub, max(1, len(ub) // 5), replace=False))
    trf = ~va

    for nm, d, sk in (('plain d2', 2, False), ('skip d2', 2, True)):
        preds = []
        for s in SEEDS_B:
            mu2, sd2 = sX[trf].mean(0), sX[trf].std(0) + 1e-9
            ym2, ys2 = sy[trf].mean(), sy[trf].std() + 1e-9
            set_seed(s)
            net = Net(sX.shape[1], d, sk, False, not sk).to(device)
            optm = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-4)
            xs = torch.tensor((sX - mu2) / sd2, dtype=torch.float, device=device)
            ysn = torch.tensor((sy - ym2) / ys2, dtype=torch.float, device=device)
            trm = torch.tensor(trf, device=device)
            vam = torch.tensor(va, device=device)
            best, bstate, since = float('inf'), None, 0
            for ep in range(EPOCHS):
                net.train(); optm.zero_grad()
                ((net(xs, sei)[trm] - ysn[trm]) ** 2).mean().backward()
                optm.step()
                if ep % 10 == 0:
                    net.eval()
                    with torch.no_grad():
                        v = float(((net(xs, sei)[vam] - ysn[vam]) ** 2).mean())
                    if v < best - 1e-5:
                        best, since = v, 0
                        bstate = {k: t.detach().clone()
                                  for k, t in net.state_dict().items()}
                    else:
                        since += 10
                        if since >= PATIENCE:
                            break
            if bstate:
                net.load_state_dict(bstate)
            net.eval()
            tmu2, tsd2 = (mu2, sd2) if regime == 'strict' else \
                         (tX.mean(0), tX.std(0) + 1e-9)
            xt = torch.tensor((tX - tmu2) / tsd2, dtype=torch.float, device=device)
            with torch.no_grad():
                preds.append(net(xt, tei).cpu().numpy() * ys2 + ym2)
        out[nm] = np.mean(preds, axis=0)

    same_zone = {src_cfg['city'], dst_cfg['city']} <= {'Bangalore', 'Hyderabad'}
    return [dict(source=f"{src_cfg['short']} {src_cfg['month'][:3]}",
                 target=f"{dst_cfg['short']} {dst_cfg['month'][:3]}",
                 pair_type='peninsular' if same_zone else 'cross-zone',
                 regime=regime, model=nm,
                 r2_absolute=r2_(ty, p),
                 r2_anomaly=r2_(ty - ty.mean(), p - p.mean()),
                 bias=float(p.mean() - ty.mean()))
            for nm, p in out.items()]


trows = []
for si, di in PAIRS:
    for regime in ('strict', 'adapted'):
        print(f"### {CONFIGS[si]['short']} -> {CONFIGS[di]['short']} "
              f"({CONFIGS[si]['month']}, {regime})", flush=True)
        trows += transfer(CONFIGS[si], CONFIGS[di], regime)

tr = pd.DataFrame(trows)
tr.to_csv(f"{OUT_ROOT}/cross_city_transfer_3city.csv", index=False)

print("\n" + "=" * 82)
print("CROSS-CITY TRANSFER — 3 cities, 12 pairs")
print("=" * 82)
for regime in ('strict', 'adapted'):
    sub = tr[tr.regime == regime]
    print(f"\n--- {regime} normalisation, R2 on spatial ANOMALY ---")
    print(sub.pivot_table(index=['source', 'target'], columns='model',
                          values='r2_anomaly').round(3).to_string())
    print("\n  mean by model:")
    print(sub.groupby('model')[['r2_absolute', 'r2_anomaly', 'bias']]
          .mean().round(3).to_string())

print("\n" + "=" * 82)
print("PENINSULAR (Blr<->Hyd) vs CROSS-ZONE (anything involving Delhi)")
print("=" * 82)
print(tr[tr.regime == 'strict']
      .pivot_table(index='model', columns='pair_type', values='r2_anomaly')
      .round(3).to_string())
print("""
  A large peninsular/cross-zone gap means generalisation is bounded by
  CLIMATE ZONE, not by city -- a more precise and more useful statement
  than 'transfer degrades', and one the two-city version could not have
  made. If the gap is small, transfer is genuinely city-agnostic within
  inland India, which is a stronger claim than the earlier run supported.
""")

s = tr[tr.regime == 'strict'].groupby('model').r2_anomaly.mean()
if 'skip d2' in s:
    best_base = s[['OLS', 'OLS+spatial lag', 'RF+spatial lag']].max()
    print(f"  skip d2 {s['skip d2']:.3f} vs best baseline {best_base:.3f} "
          f"-> {s['skip d2'] - best_base:+.3f}")
    print("  Negative means the within-city architectural advantage does not")
    print("  survive transfer. Report it; prior work largely does not test this.")

In [ ]:
# ============================================================
# IS skip ACTUALLY BETTER THAN gcnii, OR A TIE?
# ------------------------------------------------------------
# Means: skip 0.711 / gcnii 0.684 at depth 2, 0.705 / 0.691 at depth
# 6. That is a ~0.02 gap against a fold std of ~0.18, so the marginal
# means cannot settle it. Fold-to-fold variation is shared -- a hard
# fold is hard for every model -- so the PAIRED difference is the
# right statistic and its spread is far smaller than either model's
# own spread.
#
# This decides how the paper is framed:
#   skip clearly ahead  -> an architecture result, with GCNII as the
#       strongest competitor rather than plain GAT as a straw man.
#   statistical tie     -> a BENCHMARK. "Residual-style remedies
#       (skip, GCNII) hold flat with depth; JK is stable but weaker;
#       DropEdge does not prevent collapse." That is more defensible
#       anyway, since it cannot be scooped on priority.
#
# Reads remedy_comparison.csv. No refitting.
# ============================================================
import numpy as np
import pandas as pd
from scipy import stats

rm = pd.read_csv(f"{OUT_ROOT}/remedy_comparison.csv")
key = ['city', 'month', 'fold', 'depth']
piv = rm.pivot_table(index=key, columns='variant', values='r2')

VARIANTS = ['plain', 'dropedge', 'jk', 'gcnii', 'skip']

print("=" * 78)
print("PAIRED COMPARISONS (same fold, same data, difference per fold)")
print("=" * 78)
print(f"\n{'comparison':<22}{'mean diff':>11}{'sd':>9}{'wins':>9}"
      f"{'t':>8}{'p':>10}")

def paired(a, b, sub=None):
    d = (piv[a] - piv[b]) if sub is None else (piv.loc[sub, a] - piv.loc[sub, b])
    d = d.dropna()
    t, p = stats.ttest_rel(
        (piv[a] if sub is None else piv.loc[sub, a]).dropna(),
        (piv[b] if sub is None else piv.loc[sub, b]).dropna())
    return d.mean(), d.std(), int((d > 0).sum()), len(d), t, p

for a, b in [('skip', 'gcnii'), ('skip', 'jk'), ('gcnii', 'jk'),
             ('skip', 'plain'), ('gcnii', 'plain'), ('dropedge', 'plain')]:
    m, s, w, n, t, p = paired(a, b)
    print(f"{a + ' vs ' + b:<22}{m:>+11.4f}{s:>9.4f}{f'{w}/{n}':>9}"
          f"{t:>8.2f}{p:>10.4f}")

print("""
  p < 0.01 with a consistent win rate -> a real ordering.
  p > 0.05 or a win rate near half     -> a tie; report both as
      equally effective rather than picking a winner on the mean.
""")

# ---- does the answer depend on depth? ---------------------------
print("=" * 78)
print("skip vs gcnii, BY DEPTH")
print("=" * 78)
print(f"{'depth':>7}{'mean diff':>12}{'sd':>9}{'wins':>9}{'p':>10}")
for d in sorted(piv.index.get_level_values('depth').unique()):
    sub = piv.index.get_level_values('depth') == d
    m, s, w, n, t, p = paired('skip', 'gcnii', sub)
    print(f"{d:>7}{m:>+12.4f}{s:>9.4f}{f'{w}/{n}':>9}{p:>10.4f}")
print("""
  If skip leads only at shallow depth, say so -- 'skip is better at
  depth 2, equivalent from depth 4' is a precise claim and a more
  interesting one than a single averaged number.
""")

# ---- per city, to check the effect is not one-city driven -------
print("=" * 78)
print("skip vs gcnii, BY CITY")
print("=" * 78)
for city in piv.index.get_level_values('city').unique():
    sub = piv.index.get_level_values('city') == city
    m, s, w, n, t, p = paired('skip', 'gcnii', sub)
    print(f"  {city:<12}{m:>+9.4f}   wins {w}/{n}   p={p:.4f}")

# ---- collapse point ---------------------------------------------
print("\n" + "=" * 78)
print("WHERE EACH VARIANT STANDS AT DEPTH 6 (the collapse test)")
print("=" * 78)
d6 = piv.xs(6, level='depth') if 6 in piv.index.get_level_values('depth') else None
if d6 is not None:
    print(d6[VARIANTS].agg(['mean', 'std']).round(4).to_string())
    print("\n  fraction of folds with R2 < 0 (worse than predicting the mean):")
    for v in VARIANTS:
        frac = (d6[v] < 0).mean()
        print(f"    {v:<10}{frac:>6.1%}")
    print("""
  This is the sharpest single number in the paper. A variant failing
  below zero at depth 6 has not merely degraded -- it is worse than a
  constant prediction, on a task where five raw features reach R2 0.65
  with ordinary least squares.
""")